In [ ]:
import os, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# RADAR model setup
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "TrustSafeAI/RADAR-Vicuna-7B"

print(f"Loading RADAR detector: {MODEL_NAME}")
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

# load RADAR
detector = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"model loaded successfully!")

# RADAR function

def compute_radar_score(text, detector, tokenizer, device, max_len=512):

    try:

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=True
        ).to(device)

        with torch.no_grad():
            outputs = detector(**inputs)
            logits = outputs.logits

        import torch.nn.functional as F
        log_probs = F.log_softmax(logits, dim=-1)
        ai_prob = log_probs[0, 0].exp().item()

        return ai_prob

    except Exception as e:
        return np.nan

#prepare functions

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

#Evalution helper

def evaluate_radar(texts, labels, detector, tokenizer, device):

    scores = []
    for t in tqdm(texts, desc="RADAR Scoring", ncols=80):
        s = compute_radar_score(t, detector, tokenizer, device)
        scores.append(0.5 if np.isnan(s) else s)

    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }


# Beemo dataset loading

print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")

# Running all tasks

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_radar(X, y, detector, tokenizer, device)
    res[name] = mt

    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_radar.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")


# summary

summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/radar_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY - RADAR")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Loading RADAR detector: TrustSafeAI/RADAR-Vicuna-7B
Device: cuda
GPU Memory: 15.83 GB
model loaded successfully!

Loading BEEMO dataset...
Loaded 2187 samples

 H_vs_M
Samples: 4374 (2187/2187)


RADAR Scoring: 100%|████████████████████████| 4374/4374 [01:07<00:00, 64.53it/s]


AUROC: 0.5137 | Acc: 0.5007 | F1: 0.5007

 E_vs_M
Samples: 4374 (2187/2187)


RADAR Scoring: 100%|████████████████████████| 4374/4374 [01:08<00:00, 63.93it/s]


AUROC: 0.5219 | Acc: 0.5235 | F1: 0.5231

 H_vs_E
Samples: 4374 (2187/2187)


...

AUROC: 0.6269 | Acc: 0.5621 | F1: 0.6491

 G_vs_M
Samples: 8748 (6561/2187)


RADAR Scoring: 100%|████████████████████████| 8748/8748 [02:08<00:00, 68.15it/s]


AUROC: 0.3795 | Acc: 0.4321 | F1: 0.2364

 H_vs_G
Samples: 8748 (2187/6561)


RADAR Scoring: 100%|████████████████████████| 8748/8748 [02:07<00:00, 68.48it/s]


AUROC: 0.6017 | Acc: 0.5501 | F1: 0.6395

 FINAL SUMMARY - RADAR
         auroc  accuracy      f1
H_vs_M  0.5137    0.5007  0.5007
E_vs_M  0.5219    0.5235  0.5231
H_vs_E  0.4966    0.4835  0.4825
L_vs_M  0.3451    0.4152  0.2161
H_vs_L  0.6269    0.5621  0.6491
G_vs_M  0.3795    0.4321  0.2364
H_vs_G  0.6017    0.5501  0.6395

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
